In [3]:
#Import libraries
import os
import sys
sys.path.append("./scripts/")
import shutil
from brian2.units.allunits import *
from brian2.units.stdunits import *
from brian2.utils.caching import *
from scipy.sparse import coo_matrix
import numpy as np
import datalayerOmen
import random as pyrandom
import sqlalchemy as sql
import pandas as pd
from brian2 import *
from sqlalchemy import false
from sympy import true
prefs.codegen.target = "numpy"
import matplotlib.pyplot as plt
from helper import load_wmx, preprocess_monitors, generate_cue_spikes,\
                   save_vars, save_PSD, save_TFR, save_LFP, save_replay_analysis,save_wmx,save_vars_syn,SynWeightDist,save_vars_syn_cpp, _load_PF_starts 
import seaborn as sns
from sqlalchemy import create_engine
from scipy import stats
from plots import plot_violin, plot_raster, plot_posterior_trajectory, plot_PSD, plot_TFR, plot_zoomed, plot_detailed, plot_LFP, set_fig_dir, plot_wmx,set_len_sim,plot_histogram_wmx, plot_Zoom_Weights,fig_dir

In [4]:
base_path = os.path.sep.join(os.path.abspath("__file__").split(os.path.sep)[:-1])
place_cell_ratio = 0.5
PF_pklf_name = os.path.join(base_path, "files", f"PFstarts_{place_cell_ratio}_linear.pkl") if linear else None
pf_Starts = _load_PF_starts(PF_pklf_name)

In [4]:

engine = datalayerOmen.InitializeSQLEngine()
conn = engine.connect()

WARNING    /tmp/ipykernel_9713/3769643668.py:2: SAWarning: Unrecognized server version info '17.0.1110.1'.  Some SQL Server features may not function properly.
  conn = engine.connect()
 [py.warnings]


In [5]:
#Define speed using the time of the first and last spike
def speedM2 (data):
    dframe = data.copy()
    
    #dframe = dframe.sort_values (by = ['Spike_Time_int'])
    dframe = dframe.groupby (['Spike_Time_int'])
    #print(dframe)
    dframe = dframe.filter(lambda x: x['PC_ID'].count() > 2)
    dframe = dframe.groupby (['Spike_Time_int']).size().reset_index(name='count')

    #print(dframe)
    #dframe = dframe.loc(dframe['count']>=3)
   
    
    minTime = dframe['Spike_Time_int'].min()
    maxTime = dframe['Spike_Time_int'].max()
    return maxTime - minTime

In [6]:
#Plot speeds on raster plot in python
def plotSWRSlopes (expid,pcfilter,firstSWRtimestamp,secondSWRtimestamp, sqlengine):
    expid = expid
    pcfilter = pcfilter
    timeframe1 = firstSWRtimestamp
    timeframe2 = secondSWRtimestamp
    engine = sqlengine
    conn = engine.connect()
    SQLtext = ''' SELECT        [PC_ID],  Spike_Time_int
    FROM            [vwSpike_PC_Times - All PCs]
    WHERE        (expid = %d)
    ORDER BY [Spike_Time_int]
    ''' % (expid)
    data = pd.read_sql(SQLtext,conn)
    conn.close()
    data1 = data.loc[(data['PC_ID']>= pcfilter[0]) & (data['PC_ID']< pcfilter[1]) & (data['Spike_Time_int']>= timeframe1[0]) & (data['Spike_Time_int']< timeframe1[1])]
    data2 = data.loc[(data['PC_ID']>= pcfilter[0]) & (data['PC_ID']< pcfilter[1]) & (data['Spike_Time_int']>= timeframe2[0]) & (data['Spike_Time_int']< timeframe2[1])]
    data1 =data1.copy()
    data2 =data2.copy()
    data1['Series'] = 'A'
    data1['Spike_Time_int'] = data1['Spike_Time_int'] - data1['Spike_Time_int'].min()

    data2['Series'] = 'B'
    data2['Spike_Time_int'] = data2['Spike_Time_int'] - data2['Spike_Time_int'].min()
    
    data3 = pd.concat([data1,data2])
    x=data1['Spike_Time_int']
    y =data1['PC_ID']
    slope1, intercept, r, p, std_err = stats.linregress(x, y)
    x=data2['Spike_Time_int']
    y =data2['PC_ID']
    slope2, intercept, r, p, std_err = stats.linregress(x, y)
    sns.set_style('whitegrid') 
    fgrid  = sns.lmplot(x ='Spike_Time_int', y ='PC_ID', data = data3,hue='Series',markers=['x','v'], legend=False)
    fig=fgrid.fig
    fig.suptitle('Experiment: ' + str(expid))
    #fig.set(label='Experiment #: ' + str(expid))
    ax = fgrid.axes[0,0] 
    ax.legend()
    leg = ax.get_legend()
    L_labels = leg.get_texts()
    # assuming you computed r_squared which is the coefficient of determination somewhere else
    label_line_1 = r'$Slope = {0:.3f}$'.format(slope1)
    label_line_2 = r'$Slope = {0:.3f}$'.format(slope2)
    L_labels[0].set_text('A: ' + label_line_1)
    L_labels[1].set_text('B: ' + label_line_2)
    #plt.show()
    speedM21 = speedM2 (data1[['Spike_Time_int','PC_ID']])
    speedM22 = speedM2 (data2[['Spike_Time_int','PC_ID']])
    return slope1, slope2, speedM21, speedM22


In [7]:
#Plot speeds of two expriments on the same plot
def plotSWRSlopes2Exp (expid1,pcfilter1,SWRtimestamp1,expid2,pcfilter2,SWRtimestamp2, sqlengine):
    expid = expid1
    expid2 = expid2
    pcfilter = pcfilter1
    pcfilter2 = pcfilter2
    timeframe1 = SWRtimestamp1
    timeframe2 = SWRtimestamp2
    engine = sqlengine
    conn = engine.connect()
    SQLtext = ''' SELECT        [PC_ID],  Spike_Time_int
    FROM            [vwSpike_PC_Times - All PCs]
    WHERE        (expid = %d)
    ORDER BY [Spike_Time_int]
    ''' % (expid)
    data = pd.read_sql(SQLtext,conn)
    SQLtext = ''' SELECT        [PC_ID],  Spike_Time_int
    FROM            [vwSpike_PC_Times - All PCs]
    WHERE        (expid = %d)
    ORDER BY [Spike_Time_int]
    ''' % (expid2)
    data_2 = pd.read_sql(SQLtext,conn)
    conn.close()
    data1 = data.loc[(data['PC_ID']>= pcfilter[0]) & (data['PC_ID']< pcfilter[1]) & (data['Spike_Time_int']>= timeframe1[0]) & (data['Spike_Time_int']< timeframe1[1])]
    data2 = data_2.loc[(data_2['PC_ID']>= pcfilter2[0]) & (data_2['PC_ID']< pcfilter2[1]) & (data_2['Spike_Time_int']>= timeframe2[0]) & (data_2['Spike_Time_int']< timeframe2[1])]
    data1 =data1.copy()
    data2 =data2.copy()
    data1['Series'] = 'A: ' + str(expid)
    data1['Spike_Time_int'] = data1['Spike_Time_int'] - data1['Spike_Time_int'].min()
    data1['PC_ID'] = data1['PC_ID'] - data1['PC_ID'].min()

    data2['Series'] = 'A: ' + str(expid2)
    data2['Spike_Time_int'] = data2['Spike_Time_int'] - data2['Spike_Time_int'].min()
    data2['PC_ID'] = data2['PC_ID'] - data2['PC_ID'].min()

    data3 = pd.concat([data1,data2])
    x=data1['Spike_Time_int']
    y =data1['PC_ID']
    slope1, intercept, r, p, std_err = stats.linregress(x, y)
    x=data2['Spike_Time_int']
    y =data2['PC_ID']
    slope2, intercept, r, p, std_err = stats.linregress(x, y)
    sns.set_style('whitegrid') 
    fgrid  = sns.lmplot(x ='Spike_Time_int', y ='PC_ID', data = data3,hue='Series',markers=['x','v'], legend=False)
    fig=fgrid.fig
    fig.suptitle('Experiment: ' + str(expid))
    #fig.set(label='Experiment #: ' + str(expid))
    ax = fgrid.axes[0,0] 
    ax.legend()
    leg = ax.get_legend()
    L_labels = leg.get_texts()
    # assuming you computed r_squared which is the coefficient of determination somewhere else
    label_line_1 = r'$Slope = {0:.3f}$'.format(slope1)
    label_line_2 = r'$Slope = {0:.3f}$'.format(slope2)
    L_labels[0].set_text(str(expid) + ': ' + label_line_1)
    L_labels[1].set_text(str(expid2) + ': ' + label_line_2)
    speedM21 = speedM2 (data1[['Spike_Time_int','PC_ID']])
    speedM22 = speedM2 (data2[['Spike_Time_int','PC_ID']])
    #plt.show()
    return slope1, slope2, speedM21, speedM22


In [8]:
#Cycle times calculations
def calculate_cycle_times(expid, sqlengine):
    engine = sqlengine
    conn = engine.connect()
    
    # Retrieve BCs data filtered by expid
    SQLtext_BCs = '''SELECT * FROM [CUNY].[dbo].[vwSpike_PC_Times - All BCs] WHERE expid = %d''' % (expid)
    bcs_data = pd.read_sql(SQLtext_BCs, conn)
    conn.close()

    if bcs_data.empty:
        print("No data found for the specified expid.")
        return bcs_data
    
    # Sort data by BC and timestamp
    bcs_data.sort_values(by=['BC', 't'], inplace=True)

    # Create a shifted version of the t column grouped by BC
    bcs_data['Prev_t'] = bcs_data.groupby('BC')['t'].shift(1)

    # Calculate cycle times
    bcs_data['Cycle_Time'] = bcs_data['t'] - bcs_data['Prev_t']

    # Calculate Hz and replace infinite values
    bcs_data['Hz'] = 1000 / bcs_data['Cycle_Time']
    bcs_data['Hz'] = bcs_data['Hz'].replace([float('inf'), float('-inf')], 0)  # Replace inf values with 0

    return bcs_data

# Example call to the function


In [9]:
#Find BCs with high activity for replay duration
def find_high_activity_periods(expid, hz_threshold, bc_threshold, window_size, sqlengine):
    bcs_data = calculate_cycle_times(expid, sqlengine)
    empty_periods = pd.DataFrame(columns=['Start', 'End', 'Cnt', 'Duration'])
    
    if bcs_data.empty:
        return empty_periods
    
    # Filter rows where Hz is below the threshold and keep only valid times.
    filtered_data = bcs_data[bcs_data['Hz'] >= hz_threshold].copy()
    filtered_data = filtered_data.dropna(subset=['t'])
    if filtered_data.empty:
        return empty_periods
    
    # Group t in buckets of 10 seconds
    filtered_data.loc[:, 't_bucket'] = (filtered_data['t'] // window_size) * window_size
    filtered_data = filtered_data.dropna(subset=['t_bucket'])
    if filtered_data.empty:
        return empty_periods
    
    # Calculate the average count of BCs per t bucket
    avg_bc_per_bucket = filtered_data.groupby('t_bucket').size().reset_index(name='BC_Count')

    min_bucket = filtered_data['t_bucket'].min()
    max_bucket = filtered_data['t_bucket'].max()
    if pd.isna(min_bucket) or pd.isna(max_bucket):
        return empty_periods
    
    min_bucket = int(min_bucket)
    max_bucket = int(max_bucket)
    all_buckets = pd.DataFrame({'t_bucket': range(min_bucket, max_bucket + window_size, window_size)})

    # Merge with avg_bc_per_bucket to include all buckets
    avg_bc_per_bucket = all_buckets.merge(avg_bc_per_bucket, on='t_bucket', how='left').fillna(0)

    #print(avg_bc_per_bucket)
    # Set initial variables
    periods = []
    start_time = None
    end_time = None
    total_bc = 0
    count = 0
    
    # Iterate over rows to find periods where average BC count is above the threshold
    for index, row in avg_bc_per_bucket.iterrows():
        if row['BC_Count'] >= bc_threshold:
            if start_time is None:
                start_time = row['t_bucket']
            end_time = row['t_bucket'] + window_size  # End time is the end of the bucket
            total_bc += row['BC_Count']
            count += 1
        else:
            if start_time is not None:
                periods.append({
                    'Start': start_time,
                    'End': end_time,
                    'Cnt': total_bc // count,
                    'Duration': end_time - start_time
                })
                start_time = None
                end_time = None
                total_bc = 0
                count = 0   
    # Handle case where the last period extends to the end of the data
    if start_time is not None:
        periods.append({
            'Start': start_time,
            'End': end_time,
            'Cnt': total_bc // count,
            'Duration': end_time - start_time
        })
    # Convert periods to DataFrame
    
    periods_df = pd.DataFrame(periods)
    
    return periods_df
    

In [10]:
#Get spike data and calculate regression slope and speed
def fetch_spike_data(expid, sqlengine):
    engine = sqlengine
    conn = engine.connect()
    
    SQLtext = '''SELECT [PC_ID], Spike_Time_int
                 FROM [vwSpike_PC_Times - All PCs]
                 WHERE (expid = %d)
                 ORDER BY [Spike_Time_int]
              ''' % (expid)
    spike_data = pd.read_sql(SQLtext, conn)
    conn.close()
    
    return spike_data

def calculate_regression_slope(spike_data):
    # Filter data for the specified period and PC_ID range
    try:
        if spike_data.empty:
            return None, None, None, None, None
        # Check if all x values (Spike_Time_int) are identical
        if spike_data['Spike_Time_int'].nunique() == 1:
            # Return None or some default value since regression cannot be calculated
            return None, None, None, None, None
        # Calculate the regression slope
        slope, intercept, r_value, p_value, std_err = stats.linregress(spike_data['Spike_Time_int'], spike_data['PC_ID'])
        if slope is None:
            print(f"Skipping regression due to insufficient or invalid data for experiment {expid}")
            return None, None, None, None, None  # Skip to the next experiment or handle the case as needed
        return slope, intercept, r_value, p_value, std_err
    except Exception as e:
        # Catch any other unexpected errors and return None for all outputs
        print(f"Error calculating regression: {e}")
        return None, None, None, None, None

def replay_speeds(expid, sqlengine):
    hz_threshold = 120
    bc_threshold = 5    
    window_size = 10
    #min_range = 4000
    #max_range = 4500
    
    high_activity_periods = find_high_activity_periods(expid=expid, hz_threshold=hz_threshold, bc_threshold=bc_threshold, window_size=window_size, sqlengine=sqlengine)
    spike_data = fetch_spike_data(expid, sqlengine)
    Results = []
    
    for index, row in high_activity_periods.iterrows():
        start_time = row['Start']
        end_time = row['End']
        full_replay = spike_data[(spike_data['Spike_Time_int'] >= start_time) & (spike_data['Spike_Time_int'] < end_time)]
        replay_midpoint = full_replay['PC_ID'].median()
        min_range = replay_midpoint - 250
        max_range = replay_midpoint + 250
        replay_data = spike_data[(spike_data['Spike_Time_int'] >= start_time) & 
                                 (spike_data['Spike_Time_int'] < end_time) & 
                                 (spike_data['PC_ID'] >= min_range) & 
                                 (spike_data['PC_ID'] <= max_range)]
        
        slope, intercept, r_value, p_value, std_err = calculate_regression_slope(replay_data)
        duration = speedM2(replay_data)
        
        Results.append({
            'ExpID': expid,
            'Start': start_time,
            'End': end_time,
            'Slope': slope,
            'min_range': min_range,
            'max_range': max_range,
            'Duration': duration,
            'R_Value': r_value,
            'P_Value': p_value,
            'Std_Err': std_err
        })
    
    results_df = pd.DataFrame(Results) 
    results_df = results_df[(results_df['min_range'] > 1000) & (results_df['max_range'] < 7000)]
    return results_df

In [11]:
#Calculate replay speeds
def replay_speeds(expid, sqlengine):
    hz_threshold = 120
    bc_threshold = 5    #Minimum number of BCs spikes in a window with high activity (Hz >= hz_threshold)
    window_size = 10    #Window size in ms in which we are looking for high activity 
    
    # Fetch high activity periods and spike data
    high_activity_periods = find_high_activity_periods(expid=expid, hz_threshold=hz_threshold, bc_threshold=bc_threshold, window_size=window_size, sqlengine=sqlengine)
    if high_activity_periods.empty:
        print(f"No high activity periods found for expid {expid}")
        return pd.DataFrame()
    spike_data = fetch_spike_data(expid, sqlengine)
    print(f"Found {len(high_activity_periods)} high activity periods for expid {expid}")
    Results = []
    
    for index, row in high_activity_periods.iterrows():
        start_time = row['Start']
        end_time = row['End']
        
        # Get replay data
        full_replay = spike_data[(spike_data['Spike_Time_int'] >= start_time) & (spike_data['Spike_Time_int'] < end_time)]
        
        # Check if full_replay is empty
        if full_replay.empty:
            print(f"Skipping empty replay for expid {expid}, start: {start_time}, end: {end_time}")
            continue

        # Calculate replay midpoint (median PC_ID)
        replay_midpoint = full_replay['PC_ID'].median()
        
        # Check if replay_midpoint is NaN
        if pd.isna(replay_midpoint):
            print(f"Skipping replay with NaN midpoint for expid {expid}, start: {start_time}, end: {end_time}")
            continue
        
        # Calculate min and max range
        min_range = replay_midpoint - 250
        max_range = replay_midpoint + 250
        
        # Filter the replay data for the range
        replay_data = spike_data[(spike_data['Spike_Time_int'] >= start_time) & 
                                 (spike_data['Spike_Time_int'] < end_time) & 
                                 (spike_data['PC_ID'] >= min_range) & 
                                 (spike_data['PC_ID'] <= max_range)]
        
        # Skip if replay_data is empty after filtering
        if replay_data.empty:
            print(f"Skipping empty filtered replay data for expid {expid}, start: {start_time}, end: {end_time}")
            continue
        
        # Calculate regression
        slope, intercept, r_value, p_value, std_err = calculate_regression_slope(replay_data)
        
        # Handle cases where the regression fails
        if slope is None:
            print(f"Skipping invalid regression for expid {expid}, start: {start_time}, end: {end_time}")
            continue
        
        # Calculate duration (make sure speedM2 is defined correctly)
        duration = speedM2(replay_data)
        
        # Append results to the list
        Results.append({
            'ExpID': expid,
            'Start': start_time,
            'End': end_time,
            'Slope': slope,
            'min_range': min_range,
            'max_range': max_range,
            'Duration': duration,
            'R_Value': r_value,
            'P_Value': p_value,
            'Std_Err': std_err
        })
    
    # Convert results to a DataFrame
    results_df = pd.DataFrame(Results)
    
    # Ensure that min_range and max_range exist before filtering
    if 'min_range' in results_df.columns and 'max_range' in results_df.columns:
        results_df = results_df[(results_df['min_range'] > 50) & (results_df['max_range'] < 7950)]
    else:
        print(f"min_range or max_range columns missing in results_df for expid {expid}")

    return results_df


In [23]:
import pandas as pd
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, Float, text
from sqlalchemy.exc import OperationalError
from sqlalchemy import inspect
'''
def create_replays_table(engine):
    metadata = MetaData()
    replays_table = Table('replays', metadata,
        Column('ExpID', Integer),
        Column('Start', Float),
        Column('End', Float),
        Column('Slope', Float),
        Column('min_range', Float),
        Column('max_range', Float),
        Column('Duration', Float),
        Column('R_Value', Float),
        Column('P_Value', Float),
        Column('Std_Err', Float)
    )
    metadata.create_all(engine)
'''
def write_results_to_sql(results_df, expid, sqlengine):
    # Establish connection to the database
    engine = sqlengine
    conn = engine.connect()
    
    # Check if the table exists, and create it if it does not
    inspector = inspect(engine)
    if not inspector.has_table('replays'):
        create_replays_table(engine)
    
    trans = conn.begin()
    try:
        # Delete existing records with the same ExpID
        delete_query = f"DELETE FROM dbo.replays WHERE ExpID = {expid}"
        conn.execute(text(delete_query))

        if results_df.empty:
            trans.commit()
            print(f"No replay rows to write for expid {expid}")
            conn.close()
            return

        # Write the new results to the database
        results_df.to_sql('replays', con=conn, if_exists='append', index=False)

        # Commit the transaction
        trans.commit()
        print("Data committed successfully!")
    except Exception as e:
        # Rollback the transaction if any exception occurs
        trans.rollback()
        print(f"Transaction failed: {e}")
    # Delete existing records with the same ExpID
    #delete_query = f"DELETE FROM dbo.replays WHERE ExpID = {expid}"
    #print(delete_query)
    #conn.execute(text(delete_query))
    
    # Write the new results to the database
    #results_df.to_sql('replays', con=conn, if_exists='append', index=False)
    #print(results_df.shape[0], ' records written to SQL successfully.')
    #print('Data written to SQL successfully.')
    # Close the connection
    conn.close()





In [13]:
#create_replays_table(engine)

In [24]:
#Process experiments for replay table
def fetch_experiment_ids(sqlengine):
    engine = sqlengine
    conn = engine.connect()
    
    SQLtext = '''SELECT ID FROM dbo.vwExperiments where ID not in (select distinct ExpID from dbo.replays) and ID > 100400 order by ID'''
    experiment_ids = pd.read_sql(SQLtext, conn)
    conn.close()
    
    return experiment_ids['ID'].tolist()

def process_experiments(sqlengine):
    experiment_ids = fetch_experiment_ids(sqlengine)
    
    for expid in experiment_ids:
        print(expid)
        results_df = replay_speeds(expid, sqlengine)
        write_results_to_sql(results_df, expid, sqlengine)

In [25]:
process_experiments(engine)

100571
Found 1 high activity periods for expid 100571
Data committed successfully!
100572
No high activity periods found for expid 100572
No replay rows to write for expid 100572
100573
Found 6 high activity periods for expid 100573
Data committed successfully!
100574
Found 18 high activity periods for expid 100574
Data committed successfully!
100575
Found 22 high activity periods for expid 100575
Data committed successfully!
100579
Found 21 high activity periods for expid 100579
Data committed successfully!
100580
Found 16 high activity periods for expid 100580
Data committed successfully!
100581
Found 24 high activity periods for expid 100581
Data committed successfully!
100582
Found 23 high activity periods for expid 100582
Data committed successfully!
100583
Found 23 high activity periods for expid 100583
Data committed successfully!
100584
Found 23 high activity periods for expid 100584
Data committed successfully!
100585
Found 21 high activity periods for expid 100585
Data commit

In [30]:
# Env Replays: spike data joined with PC_Order from place_field_selected
def fetch_spike_data_with_order(expid, sqlengine):
    conn = sqlengine.connect()
    SQLtext = '''
        SELECT s.[PC_ID], s.[Spike_Time_int], pf.[PC_Order]
        FROM [vwSpike_PC_Times - All PCs] s
        JOIN [place_field_selected] pf
            ON s.[PC_ID] = pf.[PC_Index] AND pf.[expid] = s.[expid]
        WHERE s.[expid] = %d
        ORDER BY s.[Spike_Time_int]
    ''' % expid
    spike_data = pd.read_sql(SQLtext, conn)
    conn.close()
    return spike_data

# Calculate replay speeds using PC_Order as the spatial axis
def replay_speeds_env(expid, sqlengine):
    hz_threshold = 120
    bc_threshold = 5
    window_size = 10

    high_activity_periods = find_high_activity_periods(
        expid=expid, hz_threshold=hz_threshold, bc_threshold=bc_threshold,
        window_size=window_size, sqlengine=sqlengine
    )
    if high_activity_periods.empty:
        print(f"No high activity periods found for expid {expid}")
        return pd.DataFrame()

    spike_data = fetch_spike_data_with_order(expid, sqlengine)
    if spike_data.empty:
        print(f"No spike data with PC_Order for expid {expid}")
        return pd.DataFrame()

    max_pc_order = spike_data['PC_Order'].max()
    print(f"Found {len(high_activity_periods)} high activity periods for expid {expid}, max PC_Order: {max_pc_order}")
    Results = []

    for index, row in high_activity_periods.iterrows():
        start_time = row['Start']
        end_time = row['End']

        full_replay = spike_data[
            (spike_data['Spike_Time_int'] >= start_time) &
            (spike_data['Spike_Time_int'] < end_time)
        ]
        if full_replay.empty:
            print(f"Skipping empty replay for expid {expid}, start: {start_time}, end: {end_time}")
            continue

        replay_midpoint = full_replay['PC_Order'].median()
        if pd.isna(replay_midpoint):
            print(f"Skipping replay with NaN midpoint for expid {expid}, start: {start_time}, end: {end_time}")
            continue

        min_range = replay_midpoint - 250
        max_range = replay_midpoint + 250

        replay_data = spike_data[
            (spike_data['Spike_Time_int'] >= start_time) &
            (spike_data['Spike_Time_int'] < end_time) &
            (spike_data['PC_Order'] >= min_range) &
            (spike_data['PC_Order'] <= max_range)
        ]
        if replay_data.empty:
            print(f"Skipping empty filtered replay data for expid {expid}, start: {start_time}, end: {end_time}")
            continue

        # Rename PC_Order to PC_ID so existing helpers (calculate_regression_slope, speedM2) work unchanged
        #replay_for_regression = replay_data.rename(columns={'PC_Order': 'PC_ID'})
        replay_for_regression = replay_data.drop(columns=['PC_ID']).rename(columns={'PC_Order': 'PC_ID'})


        slope, intercept, r_value, p_value, std_err = calculate_regression_slope(replay_for_regression)
        if slope is None:
            print(f"Skipping invalid regression for expid {expid}, start: {start_time}, end: {end_time}")
            continue

        duration = speedM2(replay_for_regression)

        Results.append({
            'ExpID': expid,
            'Start': start_time,
            'End': end_time,
            'Slope': slope,
            'min_range': min_range,
            'max_range': max_range,
            'Duration': duration,
            'R_Value': r_value,
            'P_Value': p_value,
            'Std_Err': std_err
        })

    results_df = pd.DataFrame(Results)
    if not results_df.empty and 'min_range' in results_df.columns:
        results_df = results_df[
            (results_df['min_range'] > 50) & (results_df['max_range'] < max_pc_order - 50)
        ]
    return results_df

In [28]:
# Write env replay results and process all experiments
def write_results_to_env_replays(results_df, expid, sqlengine):
    conn = sqlengine.connect()
    trans = conn.begin()
    try:
        conn.execute(text(f"DELETE FROM dbo.env_replays WHERE ExpID = {expid}"))
        if results_df.empty:
            trans.commit()
            print(f"No env replay rows to write for expid {expid}")
            conn.close()
            return
        results_df.to_sql('env_replays', con=conn, if_exists='append', index=False)
        trans.commit()
        print("Env replay data committed successfully!")
    except Exception as e:
        trans.rollback()
        print(f"Transaction failed: {e}")
    conn.close()

def fetch_experiment_ids_env(sqlengine, min_expid = None, max_expid = None):
    conn = sqlengine.connect()
    SQLtext = '''SELECT ID FROM dbo.vwExperiments
                 WHERE ID NOT IN (SELECT DISTINCT ExpID FROM dbo.env_replays)'''
    if min_expid is not None:
        SQLtext += f" and ID >= {min_expid}"
    if max_expid is not None:
        SQLtext += f" and ID <= {max_expid}"
    SQLtext += " ORDER BY ID"
    experiment_ids = pd.read_sql(SQLtext, conn)
    conn.close()
    return experiment_ids['ID'].tolist()

def process_experiments_env(sqlengine, min_expid = None, max_expid = None):
    experiment_ids = fetch_experiment_ids_env(sqlengine, min_expid, max_expid)
    for expid in experiment_ids:
        print(expid)
        results_df = replay_speeds_env(expid, sqlengine)
        write_results_to_env_replays(results_df, expid, sqlengine)

In [31]:
process_experiments_env(engine, min_expid=103597)

103597
Found 41 high activity periods for expid 103597, max PC_Order: 7999
Env replay data committed successfully!
103598
Found 34 high activity periods for expid 103598, max PC_Order: 7999
Env replay data committed successfully!
103599
Found 20 high activity periods for expid 103599, max PC_Order: 7999
Env replay data committed successfully!
103600
Found 18 high activity periods for expid 103600, max PC_Order: 7999
Env replay data committed successfully!
103601
Found 43 high activity periods for expid 103601, max PC_Order: 7999
Env replay data committed successfully!
103602
Found 43 high activity periods for expid 103602, max PC_Order: 7999
Env replay data committed successfully!
103603
Found 25 high activity periods for expid 103603, max PC_Order: 7989
Env replay data committed successfully!
103604
Found 18 high activity periods for expid 103604, max PC_Order: 7992
Env replay data committed successfully!
103605
Found 31 high activity periods for expid 103605, max PC_Order: 7999
Env r

In [14]:
#Parse the experiment parameters from the JSON and write them to SQL
import pandas as pd
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, Float, String, inspect

def fetch_experiment_params(sqlengine):
    engine = sqlengine
    conn = engine.connect()
    
    SQLtext = '''SELECT ID, [Param JSON] FROM vwExperimentsAll'''
    experiment_params = pd.read_sql(SQLtext, conn)
    conn.close()
    
    return experiment_params

def process_json_params(experiment_params):
    # Create a list to store the processed data
    data = []

    for _, row in experiment_params.iterrows():
        expid = row['ID']
        json_param = row['Param JSON']
        params = json_param.split(', ')
        
        for param in params:
            if '=' in param:
                key, value = param.split('=', 1)
                key = key.strip().replace(" ", "_").lower()
                value = value.strip().replace(" ms", "")
                if value in ["True", "False"]:
                    value = value == "True"
                else:
                    try:
                        value = float(value)
                    except ValueError:
                        pass
                data.append({'ExpID': expid, 'Attribute': key, 'Value': value})
    
    # Convert the processed data into a DataFrame
    data_df = pd.DataFrame(data)
    
    # Pivot the DataFrame to get the desired column structure
    pivot_df = data_df.pivot(index='ExpID', columns='Attribute', values='Value').reset_index()
    
    return pivot_df

def create_experiments_parameters_table(engine, df):
    metadata = MetaData()
    table_name = 'Experiments Parameters'
    
    # Drop the existing table if it exists
    inspector = inspect(engine)
    if inspector.has_table(table_name):
        Table(table_name, metadata).drop(engine)
    
    # Define the table schema dynamically based on the columns present in the DataFrame
    table_columns = [Column('ExpID', Integer)]
    for column in df.columns:
        if column == 'ExpID':
            continue
        #col_type = String if column == 'cue' or column == 'eq' or column == 'on_pre' or column == 'on_post'  else Float
        col_type = Float if df[column].dtype in [float, int] else String
        #col_type = String 
        table_columns.append(Column(column, col_type))
        
    experiments_parameters_table = Table(table_name, metadata, *table_columns, extend_existing=True)
    metadata.create_all(engine)

def write_params_to_sql(processed_df, sqlengine):
    # Establish connection to the database
    engine = sqlengine
    conn = engine.connect()
    
    # Create or recreate the table schema
    create_experiments_parameters_table(engine, processed_df)
    
    # Write the new results to the database
    processed_df.to_sql('Experiments Parameters', con=conn, if_exists='append', index=False)
    
    # Close the connection
    conn.close()

# Example usage


In [ ]:
#Update the Exp paramenters table
experiment_params = fetch_experiment_params(engine)
processed_df = process_json_params(experiment_params)
processed_df['post_AuC'] = processed_df.apply(lambda row: abs(calculate_area_under_curve(row['am'], row['taum'])), axis=1)
processed_df['pre_AuC'] = processed_df.apply(lambda row: abs(calculate_area_under_curve(row['ap'], row['taup'],start=0,end=150)), axis=1)
write_params_to_sql(processed_df, engine)
print(processed_df)

In [17]:
# retrieve all experiments from experiment parameters table. Just expid and duration
def fetch_experiment_ids_and_duration(sqlengine, min_expid, max_expid):
    engine = sqlengine
    conn = engine.connect()
    
    SQLtext = '''SELECT expid, total_duration FROM dbo.[Experiments Parameters] where expid >= %d and expid <= %d''' % (min_expid, max_expid)
    experiment_ids = pd.read_sql(SQLtext, conn)
    conn.close()
    
    return experiment_ids


In [18]:
from sqlalchemy import text

def process_experiment_end_section(sqlengine, min_expid, max_expid):
    experiment_ids = fetch_experiment_ids_and_duration(sqlengine, min_expid, max_expid)
    batch_size = 25

    # Use a transaction context; it will commit on success or rollback on error
    with sqlengine.begin() as conn:
        # Delete existing rows for the range
        conn.execute(
            text("""
                DELETE FROM dbo.replays_end_section
                WHERE ExpID >= :min_expid AND ExpID <= :max_expid
            """),
            {"min_expid": min_expid, "max_expid": max_expid}
        )

        results_list = []

        for _, row in experiment_ids.iterrows():
            duration_start = float(row['total_duration']) - 3000
            duration_end = float(row['total_duration'])

            sqlquery = f"""
                SELECT * FROM dbo.spikedata
                WHERE expid = {row['expid']}
                AND spike_time >= {duration_start}
                AND previous_spike_time > 0
            """
            spike_data = pd.read_sql(sqlquery, conn)

            sqlquerybc = f"""
                SELECT * FROM dbo.BCs
                WHERE expid = {row['expid']}
                AND t >= {duration_start}
                AND previous_spike_time > 0
                AND 1000 / (t - previous_spike_time) >= 120
            """
            bc_spike_data = pd.read_sql(sqlquerybc, conn)

            spike_data = spike_data.sort_values(by=['Spike_Time'])
            bc_spike_data = bc_spike_data.sort_values(by=['t'])

            while duration_start < duration_end:
                batch_spike_data = spike_data[
                    (spike_data['Spike_Time'] >= duration_start) &
                    (spike_data['Spike_Time'] < duration_start + batch_size)
                ]
                batch_bc_spike_data = bc_spike_data[
                    (bc_spike_data['t'] >= duration_start) &
                    (bc_spike_data['t'] < duration_start + batch_size)
                ]

                if not batch_spike_data.empty:
                    if batch_spike_data['Spike_Time'].nunique() == 1:
                        print(
                            f"Skipping regression for ExpID {row['expid']}, "
                            f"Start {duration_start}: All spike times are identical"
                        )
                    else:
                        slope, intercept, r_value, p_value, std_err = stats.linregress(
                            batch_spike_data['Spike_Time'],
                            batch_spike_data['PC_ID']
                        )

                        results_list.append({
                            'ExpID': row['expid'],
                            'Start': duration_start,
                            'End': duration_start + batch_size,
                            'Slope': slope,
                            'BCs cnt': batch_bc_spike_data.shape[0],
                            'MinPC': batch_spike_data['PC_ID'].min(),
                            'MaxPC': batch_spike_data['PC_ID'].max()
                        })

                duration_start += batch_size

            print(f"Experiment {row['expid']} done")

        results = pd.DataFrame(results_list)
        if not results.empty:
            # This append is inside the same transaction as the delete
            results.to_sql('replays_end_section', con=conn, if_exists='append', index=False)
            print("Data committed successfully!")


In [20]:
process_experiment_end_section(engine, 100653, 200000)

Experiment 100653 done
Experiment 100654 done
Experiment 100655 done
Experiment 101808 done
Experiment 101809 done
Experiment 101810 done
Experiment 101811 done
Experiment 101812 done
Experiment 101813 done
Experiment 101814 done
Experiment 101815 done
Experiment 101816 done
Experiment 101817 done
Experiment 101818 done
Experiment 101819 done
Experiment 101820 done
Experiment 101821 done
Experiment 101822 done
Experiment 101823 done
Experiment 101824 done
Experiment 101825 done
Experiment 101826 done
Experiment 101827 done
Experiment 101828 done
Experiment 101829 done
Experiment 101830 done
Experiment 101831 done
Experiment 101832 done
Experiment 101833 done
Experiment 101834 done
Experiment 101835 done
Experiment 101836 done
Experiment 101837 done
Experiment 101838 done
Experiment 101839 done
Experiment 101840 done
Experiment 101841 done
Experiment 101842 done
Experiment 101843 done
Experiment 101844 done
Experiment 101845 done
Experiment 101846 done
Experiment 101847 done
Experiment 

In [32]:
def process_experiment_end_section_env(sqlengine, min_expid, max_expid):
    experiment_ids = fetch_experiment_ids_and_duration(sqlengine, min_expid, max_expid)
    batch_size = 25

    with sqlengine.begin() as conn:
        conn.execute(
            text("""
                DELETE FROM dbo.replays_end_section_env
                WHERE ExpID >= :min_expid AND ExpID <= :max_expid
            """),
            {"min_expid": min_expid, "max_expid": max_expid}
        )

        results_list = []

        for _, row in experiment_ids.iterrows():
            duration_start = float(row['total_duration']) - 3000
            duration_end = float(row['total_duration'])

            sqlquery = f"""
                SELECT s.[PC_ID], s.[Spike_Time], s.[previous_spike_time], pf.[PC_Order]
                FROM dbo.spikedata s
                JOIN dbo.place_field_selected pf
                    ON s.[PC_ID] = pf.[PC_Index] AND pf.[expid] = s.[expid]
                WHERE s.[expid] = {row['expid']}
                AND s.[spike_time] >= {duration_start}
                AND s.[previous_spike_time] > 0
            """
            spike_data = pd.read_sql(sqlquery, conn)

            sqlquerybc = f"""
                SELECT * FROM dbo.BCs
                WHERE expid = {row['expid']}
                AND t >= {duration_start}
                AND previous_spike_time > 0
                AND 1000 / (t - previous_spike_time) >= 120
            """
            bc_spike_data = pd.read_sql(sqlquerybc, conn)

            if spike_data.empty:
                print(f"No spike data with PC_Order for ExpID {row['expid']}, skipping")
                continue

            spike_data = spike_data.sort_values(by=['Spike_Time'])
            bc_spike_data = bc_spike_data.sort_values(by=['t'])

            while duration_start < duration_end:
                batch_spike_data = spike_data[
                    (spike_data['Spike_Time'] >= duration_start) &
                    (spike_data['Spike_Time'] < duration_start + batch_size)
                ]
                batch_bc_spike_data = bc_spike_data[
                    (bc_spike_data['t'] >= duration_start) &
                    (bc_spike_data['t'] < duration_start + batch_size)
                ]

                if not batch_spike_data.empty:
                    if batch_spike_data['Spike_Time'].nunique() == 1:
                        print(
                            f"Skipping regression for ExpID {row['expid']}, "
                            f"Start {duration_start}: All spike times are identical"
                        )
                    else:
                        slope, intercept, r_value, p_value, std_err = stats.linregress(
                            batch_spike_data['Spike_Time'],
                            batch_spike_data['PC_Order']
                        )

                        results_list.append({
                            'ExpID': row['expid'],
                            'Start': duration_start,
                            'End': duration_start + batch_size,
                            'Slope': slope,
                            'BCs cnt': batch_bc_spike_data.shape[0],
                            'MinPC': batch_spike_data['PC_Order'].min(),
                            'MaxPC': batch_spike_data['PC_Order'].max()
                        })

                duration_start += batch_size

            print(f"Experiment {row['expid']} done")

        results = pd.DataFrame(results_list)
        if not results.empty:
            results.to_sql('replays_end_section_env', con=conn, if_exists='append', index=False)
            print("Data committed successfully!")

In [34]:
process_experiment_end_section_env(engine, 103591, 200000)

Experiment 103591 done
No spike data with PC_Order for ExpID 103592, skipping
No spike data with PC_Order for ExpID 103593, skipping
Experiment 103594 done
No spike data with PC_Order for ExpID 103595, skipping
No spike data with PC_Order for ExpID 103596, skipping
Experiment 103597 done
Experiment 103598 done
No spike data with PC_Order for ExpID 103599, skipping
No spike data with PC_Order for ExpID 103600, skipping
Experiment 103601 done
Experiment 103602 done
No spike data with PC_Order for ExpID 103603, skipping
No spike data with PC_Order for ExpID 103604, skipping
Experiment 103605 done
Experiment 103606 done
Experiment 103607 done
Experiment 103608 done
Experiment 103609 done
No spike data with PC_Order for ExpID 103610, skipping
No spike data with PC_Order for ExpID 103611, skipping
Experiment 103612 done
Experiment 103613 done
No spike data with PC_Order for ExpID 103614, skipping
No spike data with PC_Order for ExpID 103615, skipping
Experiment 103616 done
Experiment 103617 